# 7. Machine learning models

Notebook 06 established the LEAR as a strong linear baseline. This notebook
introduces two machine-learning models -- **LightGBM quantile regression**
and a **PyTorch day-ahead quantile network** -- that can capture the
nonlinear price dynamics LEAR misses.

Both models produce a full quantile fan (not just a point forecast),
enabling probabilistic dispatch. We compare all three approaches on
forecast accuracy *and* battery revenue.

## Objectives

- Train a LightGBM quantile model and interpret its feature importance.
- Visualise quantile fan forecasts for sample days.
- Train a neural network (DayAheadQuantileNet) on the arcsinh target with pinball loss.
- Plot learning curves to diagnose over/underfitting.
- Build a comparison table: LEAR vs GBT vs NN (MAE, rMAE, CRPS, pinball by quantile).
- Dispatch with MPC using the median and mean of the quantile fan; report capture ratio.
- Assemble a full scorecard across all models.

## Prerequisites

- Notebook 05: feature matrix built.
- Notebook 06: LEAR baseline fitted and evaluated.
- Processed parquet file `data/processed/SA1_30min.parquet`.
- LightGBM, PyTorch installed (`pip install lightgbm torch`).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from grian.config import load_config, repo_root
from grian.dispatch import capture_ratio, schedule
from grian.features import build_matrix
from grian.metrics import crps, mae, pinball_loss, relative_mae
from grian.models.baselines import similar_day_naive
from grian.models.gbt import GBTQuantile
from grian.models.lear import LEAR
from grian.models.nn import DayAheadQuantileNet, pinball_loss_fn
from grian.viz import apply_style, save_fig

warnings.filterwarnings("ignore", category=UserWarning)

cfg = load_config()
apply_style()

REGION = cfg["region"]
SEED = cfg["seed"]
QUANTILES = cfg["quantiles"]
HORIZON = cfg["horizon_periods"]

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Region: {REGION}")
print(f"Quantiles: {QUANTILES}")
print(f"Horizon: {HORIZON} periods")

---
## Data preparation

Load the 30-minute dataset, build the feature matrix, and split into
train/test. The target is `arcsinh(price)` -- this stabilises the
variance of NEM prices (which can spike to $15,000/MWh) and is
inverted with `sinh` before scoring in $/MWh.

In [ ]:
df = pd.read_parquet(repo_root() / "data" / "processed" / f"{REGION}_30min.parquet")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

In [ ]:
df[["price", "demand"]].describe()

In [ ]:
# Build feature matrix
X = build_matrix(df[["price"]], df[["demand"]])
y = np.arcsinh(df["price"]).reindex(X.index)

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")
X.head()

In [ ]:
# Train/test split using config dates
train_start = cfg["train_start"]
train_end = cfg["train_end"]
test_start = cfg["test_start"]
test_end = cfg["test_end"]

X_train = X.loc[train_start:train_end]
y_train = y.loc[train_start:train_end]
X_test = X.loc[test_start:test_end]
y_test = y.loc[test_start:test_end]

print(f"Train: {len(X_train):,} rows  ({train_start} to {train_end})")
print(f"Test:  {len(X_test):,} rows  ({test_start} to {test_end})")

---
## 1. LightGBM quantile regression

Gradient-boosted trees are the workhorse of tabular ML. Here we train
one LightGBM model per quantile level, producing a full predictive
distribution. The GBT can capture nonlinear interactions (e.g. demand
x time-of-day) that the LEAR's linear structure cannot.

In [ ]:
gbt = GBTQuantile(
    quantiles=QUANTILES,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    seed=SEED,
)

print("Fitting GBT quantile models (one per quantile level)...")
gbt.fit(X_train, y_train)
print(f"Fitted {len(gbt.models_)} quantile models, {gbt.n_estimators} trees each")

In [ ]:
# Predict on test set (in arcsinh space) and invert to $/MWh
gbt_pred_asinh = gbt.predict(X_test)
gbt_pred = gbt_pred_asinh.apply(np.sinh)

print(f"GBT predictions shape: {gbt_pred.shape}")
gbt_pred.head()

### Feature importance

LightGBM tracks how often each feature is used for splitting and how
much it reduces the loss. This tells us which inputs the model finds
most useful -- compare this later with LEAR's LASSO selection.

In [ ]:
fi = gbt.feature_importance()
print(fi.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fi["feature"], fi["importance"], color="steelblue")
ax.set_xlabel("Importance (mean split gain)")
ax.set_title("GBT feature importance (averaged across quantile models)")
ax.invert_yaxis()
fig.tight_layout()
save_fig(fig, "gbt_feature_importance")
plt.show()

---
## 2. Quantile fan plots for sample days

The GBT produces seven quantile levels for every half-hour. Plotting
these as a shaded fan shows the model's uncertainty: wider fans mean
the model is less confident about prices in that period.

In [ ]:
# Invert test targets to $/MWh for plotting
y_actual = np.sinh(y_test)

# Pick three sample days: calm, volatile, and a weekend
daily_range = y_actual.groupby(y_actual.index.date).apply(
    lambda s: s.max() - s.min()
)
volatile_day = pd.Timestamp(daily_range.idxmax())
calm_day = pd.Timestamp(daily_range.idxmin())

test_dates_all = pd.date_range(test_start, test_end, freq="D")
weekend_days = [d for d in test_dates_all if d.dayofweek >= 5]
weekend_day = weekend_days[len(weekend_days) // 2]

sample_days = [
    (calm_day, "Calm day"),
    (volatile_day, "Volatile day"),
    (weekend_day, "Weekend"),
]

print("Sample days:")
for day, label in sample_days:
    print(f"  {label}: {day.date()} ({day.day_name()})")

In [ ]:
quantile_cols = [f"q{q:.2f}" for q in QUANTILES]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, (day, label) in zip(axes, sample_days):
    day_end = day + pd.Timedelta(hours=23, minutes=30)
    mask = (gbt_pred.index >= day) & (gbt_pred.index <= day_end)
    pred_day = gbt_pred.loc[mask]
    actual_day = y_actual.loc[mask]

    if len(pred_day) < HORIZON:
        ax.set_title(f"{label} (insufficient data)")
        continue

    hours = np.arange(len(pred_day)) / 2
    n_q = len(QUANTILES)

    # Shade quantile bands (symmetric pairs from outer to inner)
    for i in range(n_q // 2):
        lower = pred_day[quantile_cols[i]].values
        upper = pred_day[quantile_cols[n_q - 1 - i]].values
        ax.fill_between(
            hours, lower, upper,
            alpha=0.15 + 0.1 * i, color="steelblue",
            label=f"q{QUANTILES[i]:.2f}--q{QUANTILES[n_q-1-i]:.2f}" if i == 0 else None,
        )

    ax.plot(hours, pred_day["q0.50"].values, color="steelblue",
            linewidth=1.5, label="Median (q0.50)")
    ax.plot(hours, actual_day.values[:len(hours)], color="black",
            linewidth=1.5, label="Actual", linestyle="--")

    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Price ($/MWh)")
    ax.set_title(f"{label}\n{day.date()}")
    ax.legend(fontsize=8)

fig.suptitle("GBT quantile fan forecasts -- sample test days", fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, "gbt_quantile_fans")
plt.show()

---
## 3. Neural network: DayAheadQuantileNet

The GBT predicts each half-hour independently. A neural network can
learn joint structure across the full day: the shape of the daily price
profile, correlations between morning and evening periods, etc.

The `DayAheadQuantileNet` takes a single feature vector (from the
forecast origin at the start of the day) and outputs all 48 half-hour
quantiles at once.

**Data structure:** Each sample is one day. The feature vector is the
row from the feature matrix at the first period of the day (00:00).
The target is the 48-period price profile for that day, in arcsinh space.

In [ ]:
def make_daily_dataset(X_df, y_series, horizon=48):
    """Group feature matrix and target into daily samples.

    Each sample: feature vector from first period of the day -> horizon targets.
    Returns (X_days, y_days, day_dates).
    """
    X_list, y_list, dates = [], [], []
    for date, group in y_series.groupby(y_series.index.date):
        if len(group) < horizon:
            continue
        day_start = group.index[0]
        if day_start not in X_df.index:
            continue
        X_list.append(X_df.loc[day_start].values)
        y_list.append(group.iloc[:horizon].values)
        dates.append(date)
    return np.array(X_list), np.array(y_list), dates


X_train_daily, y_train_daily, train_dates = make_daily_dataset(X_train, y_train, HORIZON)
X_test_daily, y_test_daily, test_dates_list = make_daily_dataset(X_test, y_test, HORIZON)

print(f"Training days: {len(X_train_daily)}")
print(f"Test days:     {len(X_test_daily)}")
print(f"Features per day: {X_train_daily.shape[1]}")
print(f"Targets per day:  {y_train_daily.shape[1]}")

In [ ]:
# Split training into fit / validation (last 20%)
n_total = len(X_train_daily)
n_val = int(n_total * 0.2)
n_fit = n_total - n_val

X_fit = X_train_daily[:n_fit]
y_fit = y_train_daily[:n_fit]
X_val = X_train_daily[n_fit:]
y_val = y_train_daily[n_fit:]

print(f"Fit:        {n_fit} days")
print(f"Validation: {n_val} days")

In [ ]:
# Select device: MPS if available, else CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# Convert to tensors
X_fit_t = torch.tensor(X_fit, dtype=torch.float32)
y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test_daily, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test_daily, dtype=torch.float32).to(device)

quantiles_t = torch.tensor(QUANTILES, dtype=torch.float32).to(device)

# DataLoader for mini-batch training
train_ds = TensorDataset(X_fit_t, y_fit_t)
train_loader = DataLoader(
    train_ds, batch_size=64, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

In [ ]:
# Build the model
n_features = X_fit.shape[1]
n_quantiles = len(QUANTILES)

model_nn = DayAheadQuantileNet(
    n_features=n_features,
    horizon=HORIZON,
    n_quantiles=n_quantiles,
    hidden_dims=(256, 128, 64),
).to(device)

n_params = sum(p.numel() for p in model_nn.parameters())
print(model_nn)
print(f"\nTotal parameters: {n_params:,}")

In [ ]:
# Training loop with early stopping
optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3)
patience = 20
max_epochs = 300

train_losses = []
val_losses = []
best_val_loss = float("inf")
best_epoch = 0
best_state = None
wait = 0

for epoch in range(max_epochs):
    # --- Train ---
    model_nn.train()
    epoch_loss = 0.0
    n_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model_nn(xb)  # (batch, horizon, n_quantiles)
        actual = yb.unsqueeze(-1)  # (batch, horizon, 1)
        loss = pinball_loss_fn(pred, actual, quantiles_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_losses.append(epoch_loss / n_batches)

    # --- Validate ---
    model_nn.eval()
    with torch.no_grad():
        val_pred = model_nn(X_val_t)
        val_actual = y_val_t.unsqueeze(-1)
        val_loss = pinball_loss_fn(val_pred, val_actual, quantiles_t).item()
    val_losses.append(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model_nn.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    if (epoch + 1) % 25 == 0:
        print(
            f"Epoch {epoch + 1:3d}  train_loss={train_losses[-1]:.4f}  "
            f"val_loss={val_loss:.4f}  best={best_val_loss:.4f} (ep {best_epoch + 1})"
        )

# Restore best model
model_nn.load_state_dict(best_state)
model_nn.to(device)
print(f"\nRestored best model from epoch {best_epoch + 1} "
      f"(val_loss={best_val_loss:.4f})")

---
## 4. Learning curves

The gap between train and validation loss reveals whether the model is
over- or underfitting. A closing gap means the model generalises well;
a widening gap means it is memorising the training data.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
epochs_range = np.arange(1, len(train_losses) + 1)
ax.plot(epochs_range, train_losses, label="Train loss", linewidth=1.5)
ax.plot(epochs_range, val_losses, label="Validation loss", linewidth=1.5)
ax.axvline(best_epoch + 1, color="grey", linestyle="--", linewidth=1,
           label=f"Best epoch ({best_epoch + 1})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Pinball loss (arcsinh scale)")
ax.set_title("NN learning curves")
ax.legend()
fig.tight_layout()
save_fig(fig, "nn_learning_curves")
plt.show()

### NN predictions on test set

Generate predictions for the full test period. The NN outputs shape
`(n_days, 48, 7)` -- one 48-period quantile fan per day. We invert
the arcsinh transform to get predictions in $/MWh.

In [ ]:
model_nn.eval()
with torch.no_grad():
    nn_pred_asinh_t = model_nn(X_test_t)  # (n_days, horizon, n_quantiles)

nn_pred_asinh = nn_pred_asinh_t.cpu().numpy()
nn_pred = np.sinh(nn_pred_asinh)  # invert to $/MWh

print(f"NN prediction shape: {nn_pred.shape}")
print("  (days, horizon, quantiles)")

In [ ]:
# NN fan plots for the same sample days
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, (day, label) in zip(axes, sample_days):
    day_date = day.date() if hasattr(day, "date") else day
    if day_date not in test_dates_list:
        ax.set_title(f"{label} (not in test set)")
        continue
    day_idx = test_dates_list.index(day_date)
    pred_day = nn_pred[day_idx]  # (48, 7)

    # Actual prices for this day
    day_start = pd.Timestamp(day_date)
    day_end_ts = day_start + pd.Timedelta(hours=23, minutes=30)
    actual_day = y_actual.loc[day_start:day_end_ts].values[:HORIZON]

    hours = np.arange(HORIZON) / 2
    n_q = len(QUANTILES)

    for i in range(n_q // 2):
        ax.fill_between(
            hours, pred_day[:, i], pred_day[:, n_q - 1 - i],
            alpha=0.15 + 0.1 * i, color="darkorange",
        )

    median_idx = QUANTILES.index(0.5)
    ax.plot(hours, pred_day[:, median_idx], color="darkorange",
            linewidth=1.5, label="NN median")
    ax.plot(hours, actual_day, color="black",
            linewidth=1.5, label="Actual", linestyle="--")

    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Price ($/MWh)")
    ax.set_title(f"{label}\n{day_date}")
    ax.legend(fontsize=8)

fig.suptitle("NN quantile fan forecasts -- sample test days", fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, "nn_quantile_fans")
plt.show()

---
## 5. Comparison table: LEAR vs GBT vs NN

We now fit the LEAR baseline so all three models are scored on exactly
the same test periods. All metrics are computed in $/MWh after inverting
the arcsinh transform.

Metrics:
- **MAE**: point forecast accuracy (using the median quantile for GBT/NN,
  point prediction for LEAR).
- **rMAE**: relative to the similar-day naive baseline.
- **CRPS**: overall probabilistic calibration.
- **Pinball loss per quantile**: sharpness at each level.

In [ ]:
# Fit LEAR on training data (per-hour model, as in NB06)
lear = LEAR(per_hour=True, seed=SEED)
lear.fit(X_train, y_train)

# LEAR point predictions on test (inverted to $/MWh)
lear_pred_asinh = lear.predict(X_test)
lear_pred = np.sinh(lear_pred_asinh)

print(f"LEAR test predictions: {len(lear_pred):,} periods")

In [ ]:
# Generate naive baseline for rMAE calculation
naive_preds = pd.Series(dtype=float, name="naive")
for day in pd.date_range(test_start, test_end, freq="D"):
    try:
        naive_fc = similar_day_naive(df[["price"]], origin=day, horizon=48)
        naive_preds = pd.concat([naive_preds, naive_fc])
    except Exception:
        continue

print(f"Naive predictions: {len(naive_preds):,} periods")

In [ ]:
# Flatten NN predictions to a per-period DataFrame for scoring
nn_rows = []
for i, day_date in enumerate(test_dates_list):
    day_start = pd.Timestamp(day_date)
    idx = pd.date_range(day_start, periods=HORIZON, freq="30min")
    for t in range(HORIZON):
        row = {"timestamp": idx[t]}
        for q_idx, q in enumerate(QUANTILES):
            row[f"q{q:.2f}"] = nn_pred[i, t, q_idx]
        nn_rows.append(row)

nn_pred_df = pd.DataFrame(nn_rows).set_index("timestamp")
nn_pred_df = nn_pred_df[~nn_pred_df.index.duplicated(keep="first")]

print(f"NN flattened predictions: {len(nn_pred_df):,} periods")
nn_pred_df.head()

In [ ]:
# Common index across all models
common = (
    y_actual.index
    .intersection(gbt_pred.index)
    .intersection(nn_pred_df.index)
    .intersection(lear_pred.index)
    .intersection(naive_preds.index)
)
print(f"Common scoring periods: {len(common):,}")

y_act_c = y_actual.loc[common].values
y_naive_c = naive_preds.loc[common].values

In [ ]:
# --- LEAR scores ---
lear_median_c = lear_pred.loc[common].values
mae_lear = mae(y_act_c, lear_median_c)
rmae_lear = relative_mae(y_act_c, lear_median_c, y_naive_c)

# --- GBT scores ---
gbt_c = gbt_pred.loc[common]
gbt_median_c = gbt_c["q0.50"].values
mae_gbt = mae(y_act_c, gbt_median_c)
rmae_gbt = relative_mae(y_act_c, gbt_median_c, y_naive_c)
gbt_qf = gbt_c[[f"q{q:.2f}" for q in QUANTILES]].values
crps_gbt = crps(y_act_c, gbt_qf, np.array(QUANTILES))

# --- NN scores ---
nn_c = nn_pred_df.loc[common]
nn_median_c = nn_c["q0.50"].values
mae_nn = mae(y_act_c, nn_median_c)
rmae_nn = relative_mae(y_act_c, nn_median_c, y_naive_c)
nn_qf = nn_c[[f"q{q:.2f}" for q in QUANTILES]].values
crps_nn = crps(y_act_c, nn_qf, np.array(QUANTILES))

# --- Naive MAE ---
mae_naive = mae(y_act_c, y_naive_c)

print(f"{'Model':<15} {'MAE ($/MWh)':>12} {'rMAE':>8} {'CRPS ($/MWh)':>13}")
print("-" * 50)
print(f"{'Naive':<15} ${mae_naive:>10.2f} {'1.000':>8} {'--':>13}")
print(f"{'LEAR':<15} ${mae_lear:>10.2f} {rmae_lear:>8.3f} {'--':>13}")
print(f"{'GBT':<15} ${mae_gbt:>10.2f} {rmae_gbt:>8.3f} ${crps_gbt:>11.2f}")
print(f"{'NN':<15} ${mae_nn:>10.2f} {rmae_nn:>8.3f} ${crps_nn:>11.2f}")

In [ ]:
# Pinball loss by quantile level for GBT and NN
pb_gbt = [
    pinball_loss(y_act_c, gbt_qf[:, i], QUANTILES[i])
    for i in range(len(QUANTILES))
]
pb_nn = [
    pinball_loss(y_act_c, nn_qf[:, i], QUANTILES[i])
    for i in range(len(QUANTILES))
]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(QUANTILES))
width = 0.35
ax.bar(x - width / 2, pb_gbt, width, label="GBT", color="steelblue")
ax.bar(x + width / 2, pb_nn, width, label="NN", color="darkorange")
ax.set_xlabel("Quantile level")
ax.set_ylabel("Pinball loss ($/MWh)")
ax.set_title("Pinball loss by quantile: GBT vs NN")
ax.set_xticks(x)
ax.set_xticklabels([f"{q:.2f}" for q in QUANTILES])
ax.legend()
fig.tight_layout()
save_fig(fig, "pinball_by_quantile")
plt.show()

# Print table
print(f"{'Quantile':<10} {'GBT':>12} {'NN':>12}")
print("-" * 36)
for i, q in enumerate(QUANTILES):
    print(f"{q:<10.2f} ${pb_gbt[i]:>10.2f} ${pb_nn[i]:>10.2f}")

In [ ]:
# MAE by hour of day: all three models vs naive
hours_c = common.hour
mae_by_h = {"LEAR": [], "GBT": [], "NN": [], "Naive": []}
for h in range(24):
    m = hours_c == h
    mae_by_h["LEAR"].append(mae(y_act_c[m], lear_median_c[m]))
    mae_by_h["GBT"].append(mae(y_act_c[m], gbt_median_c[m]))
    mae_by_h["NN"].append(mae(y_act_c[m], nn_median_c[m]))
    mae_by_h["Naive"].append(mae(y_act_c[m], y_naive_c[m]))

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(24)
width = 0.2
ax.bar(x - 1.5 * width, mae_by_h["Naive"], width, label="Naive", color="lightgrey")
ax.bar(x - 0.5 * width, mae_by_h["LEAR"], width, label="LEAR", color="coral")
ax.bar(x + 0.5 * width, mae_by_h["GBT"], width, label="GBT", color="steelblue")
ax.bar(x + 1.5 * width, mae_by_h["NN"], width, label="NN", color="darkorange")
ax.set_xlabel("Hour of day")
ax.set_ylabel("MAE ($/MWh)")
ax.set_title("MAE by hour: all models")
ax.set_xticks(x)
ax.legend()
fig.tight_layout()
save_fig(fig, "mae_by_hour_all_models")
plt.show()

---
## 6. Dispatch: forecast-driven battery revenue

A forecast is only as good as the money it makes. For each day in the
test period, we:
1. Take the model's forecast (using the median and the mean of the quantile fan).
2. Solve the battery LP to get the optimal charge/discharge schedule.
3. Compute revenue against *actual* prices.
4. Compare to perfect foresight.

We test two dispatch strategies:
- **Median dispatch**: use the q0.50 forecast as the price signal.
- **Mean dispatch**: use the mean of all quantiles as the price signal
  (upweights tail risk from price spikes).

In [ ]:
battery_kwargs = {
    "power_mw": cfg["battery"]["power_mw"],
    "duration_hours": cfg["battery"]["duration_hours"],
    "efficiency": cfg["battery"]["efficiency_roundtrip"],
    "max_cycles": cfg["battery"]["max_cycles_per_day"],
}
print("Battery parameters:")
for k, v in battery_kwargs.items():
    print(f"  {k}: {v}")

In [ ]:
# Daily dispatch for all models
dispatch = {
    "LEAR": {"revenue": [], "days": []},
    "GBT_median": {"revenue": [], "days": []},
    "GBT_mean": {"revenue": [], "days": []},
    "NN_median": {"revenue": [], "days": []},
    "NN_mean": {"revenue": [], "days": []},
    "perfect": {"revenue": [], "days": []},
    "naive": {"revenue": [], "days": []},
}


def dispatch_and_score(forecast_prices, actual_prices):
    """Dispatch against forecast, compute revenue against actual prices."""
    res = schedule(forecast_prices, **battery_kwargs)
    if res["status"] != "optimal":
        return 0.0
    net_power = res["discharge"] - res["charge"]
    return float(np.sum(actual_prices * net_power * 0.5))


for day in pd.date_range(test_start, test_end, freq="D"):
    day_end = day + pd.Timedelta(hours=23, minutes=30)

    actual_day = y_actual.loc[day:day_end]
    if len(actual_day) < HORIZON:
        continue
    actual_48 = actual_day.iloc[:HORIZON].values

    # Perfect foresight
    res_perfect = schedule(actual_48, **battery_kwargs)
    if res_perfect["status"] != "optimal":
        continue

    dispatch["perfect"]["revenue"].append(res_perfect["revenue"])
    dispatch["perfect"]["days"].append(day)

    # --- LEAR ---
    lear_day = lear_pred.reindex(actual_day.index)
    if not lear_day.isna().any() and len(lear_day) >= HORIZON:
        dispatch["LEAR"]["revenue"].append(
            dispatch_and_score(lear_day.iloc[:HORIZON].values, actual_48)
        )
        dispatch["LEAR"]["days"].append(day)

    # --- GBT median and mean ---
    gbt_day = gbt_pred.reindex(actual_day.index)
    if not gbt_day.isna().any().any() and len(gbt_day) >= HORIZON:
        gbt_day_48 = gbt_day.iloc[:HORIZON]
        dispatch["GBT_median"]["revenue"].append(
            dispatch_and_score(gbt_day_48["q0.50"].values, actual_48)
        )
        dispatch["GBT_median"]["days"].append(day)
        dispatch["GBT_mean"]["revenue"].append(
            dispatch_and_score(gbt_day_48.values.mean(axis=1), actual_48)
        )
        dispatch["GBT_mean"]["days"].append(day)

    # --- NN median and mean ---
    nn_day = nn_pred_df.reindex(actual_day.index)
    if not nn_day.isna().any().any() and len(nn_day) >= HORIZON:
        nn_day_48 = nn_day.iloc[:HORIZON]
        dispatch["NN_median"]["revenue"].append(
            dispatch_and_score(nn_day_48["q0.50"].values, actual_48)
        )
        dispatch["NN_median"]["days"].append(day)
        dispatch["NN_mean"]["revenue"].append(
            dispatch_and_score(nn_day_48.values.mean(axis=1), actual_48)
        )
        dispatch["NN_mean"]["days"].append(day)

    # --- Naive ---
    naive_day = naive_preds.reindex(actual_day.index)
    if not naive_day.isna().any() and len(naive_day) >= HORIZON:
        dispatch["naive"]["revenue"].append(
            dispatch_and_score(naive_day.iloc[:HORIZON].values, actual_48)
        )
        dispatch["naive"]["days"].append(day)

for name in dispatch:
    n = len(dispatch[name]["revenue"])
    total = sum(dispatch[name]["revenue"])
    print(f"{name:<15} {n:>4} days  total=${total:>12,.0f}")

In [ ]:
# Capture ratios
total_perfect = sum(dispatch["perfect"]["revenue"])

cr = {}
for name in ["LEAR", "GBT_median", "GBT_mean", "NN_median", "NN_mean", "naive"]:
    total = sum(dispatch[name]["revenue"])
    cr[name] = capture_ratio(total, total_perfect)

print(f"{'Model':<15} {'Total revenue':>15} {'Capture ratio':>15}")
print("-" * 47)
for name in ["naive", "LEAR", "GBT_median", "GBT_mean", "NN_median", "NN_mean"]:
    total = sum(dispatch[name]["revenue"])
    print(f"{name:<15} ${total:>13,.0f} {cr[name]:>14.1%}")
print(f"{'Perfect':<15} ${total_perfect:>13,.0f} {'100.0%':>14}")

In [ ]:
# Cumulative revenue plot
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    dispatch["perfect"]["days"],
    np.cumsum(dispatch["perfect"]["revenue"]),
    label="Perfect foresight", linewidth=1.5, color="grey", linestyle="--",
)

line_styles = {
    "naive": ("lightgrey", "-"),
    "LEAR": ("coral", "-"),
    "GBT_median": ("steelblue", "-"),
    "GBT_mean": ("royalblue", "--"),
    "NN_median": ("darkorange", "-"),
    "NN_mean": ("goldenrod", "--"),
}

for name, (color, ls) in line_styles.items():
    if len(dispatch[name]["revenue"]) > 0:
        ax.plot(
            dispatch[name]["days"],
            np.cumsum(dispatch[name]["revenue"]),
            label=f"{name} (CR={cr[name]:.1%})",
            linewidth=1.5, color=color, linestyle=ls,
        )

ax.set_xlabel("Date")
ax.set_ylabel("Cumulative revenue ($)")
ax.set_title("Cumulative battery revenue: all models")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
fig.tight_layout()
save_fig(fig, "ml_cumulative_revenue")
plt.show()

In [ ]:
# Compare median vs mean dispatch strategy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GBT: median vs mean
if dispatch["GBT_median"]["revenue"] and dispatch["GBT_mean"]["revenue"]:
    axes[0].scatter(
        dispatch["GBT_median"]["revenue"],
        dispatch["GBT_mean"]["revenue"],
        alpha=0.3, s=10, color="steelblue",
    )
    lim = max(
        max(np.abs(dispatch["GBT_median"]["revenue"])),
        max(np.abs(dispatch["GBT_mean"]["revenue"])),
    )
    axes[0].plot([-lim, lim], [-lim, lim], "--", color="grey", linewidth=1)
    axes[0].set_xlabel("GBT median dispatch ($/day)")
    axes[0].set_ylabel("GBT mean dispatch ($/day)")
    axes[0].set_title(
        f"GBT: median ({cr['GBT_median']:.1%}) vs mean ({cr['GBT_mean']:.1%})"
    )

# NN: median vs mean
if dispatch["NN_median"]["revenue"] and dispatch["NN_mean"]["revenue"]:
    axes[1].scatter(
        dispatch["NN_median"]["revenue"],
        dispatch["NN_mean"]["revenue"],
        alpha=0.3, s=10, color="darkorange",
    )
    lim = max(
        max(np.abs(dispatch["NN_median"]["revenue"])),
        max(np.abs(dispatch["NN_mean"]["revenue"])),
    )
    axes[1].plot([-lim, lim], [-lim, lim], "--", color="grey", linewidth=1)
    axes[1].set_xlabel("NN median dispatch ($/day)")
    axes[1].set_ylabel("NN mean dispatch ($/day)")
    axes[1].set_title(
        f"NN: median ({cr['NN_median']:.1%}) vs mean ({cr['NN_mean']:.1%})"
    )

fig.suptitle(
    "Median vs mean dispatch strategy (daily revenue)", fontsize=13, y=1.02
)
fig.tight_layout()
save_fig(fig, "median_vs_mean_dispatch")
plt.show()

---
## 7. Full scorecard

Pulling everything together into a single scorecard: forecast accuracy
metrics and dispatch revenue for all models.

In [ ]:
scorecard = pd.DataFrame({
    "Model": ["Naive", "LEAR", "GBT", "NN"],
    "MAE ($/MWh)": [mae_naive, mae_lear, mae_gbt, mae_nn],
    "rMAE": [1.0, rmae_lear, rmae_gbt, rmae_nn],
    "CRPS ($/MWh)": [np.nan, np.nan, crps_gbt, crps_nn],
    "CR (median)": [
        cr.get("naive", np.nan),
        cr.get("LEAR", np.nan),
        cr.get("GBT_median", np.nan),
        cr.get("NN_median", np.nan),
    ],
    "CR (mean)": [
        np.nan,
        np.nan,
        cr.get("GBT_mean", np.nan),
        cr.get("NN_mean", np.nan),
    ],
})
scorecard = scorecard.set_index("Model")
scorecard

In [ ]:
# Visual scorecard: MAE and capture ratio side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_list = ["Naive", "LEAR", "GBT", "NN"]
bar_colors = ["lightgrey", "coral", "steelblue", "darkorange"]

# MAE bars
axes[0].bar(models_list, scorecard["MAE ($/MWh)"].values, color=bar_colors)
axes[0].set_ylabel("MAE ($/MWh)")
axes[0].set_title("Point forecast accuracy (lower is better)")
for i, v in enumerate(scorecard["MAE ($/MWh)"].values):
    axes[0].text(i, v + 0.5, f"${v:.1f}", ha="center", fontsize=10)

# Capture ratio bars (median strategy)
cr_vals = scorecard["CR (median)"].values
axes[1].bar(models_list, cr_vals * 100, color=bar_colors)
axes[1].set_ylabel("Capture ratio (%)")
axes[1].set_title("Battery dispatch value (higher is better)")
for i, v in enumerate(cr_vals):
    if not np.isnan(v):
        axes[1].text(i, v * 100 + 0.5, f"{v:.1%}", ha="center", fontsize=10)

fig.tight_layout()
save_fig(fig, "ml_scorecard")
plt.show()

---
## Exercises

### Exercise 1: Feature importance alignment -- GBT vs LEAR

The GBT ranks features by split gain. LEAR selects features via LASSO
sparsity. Do they agree on what matters? Compute the Spearman rank
correlation between GBT feature importance and LEAR's feature selection
frequency across slots.

<details><summary>Hint 1</summary>
For LEAR, count how many of the 48 slots select each feature (using
<code>lear.selected_features()</code>). This gives a "selection frequency"
per feature.
</details>

<details><summary>Hint 2</summary>
For GBT, use <code>gbt.feature_importance()</code> to get importance per
feature. Merge on feature name and compute Spearman's rho.
</details>

<details><summary>Hint 3</summary>
Use <code>scipy.stats.spearmanr</code> to compute the rank correlation.
A correlation near 1 means both models agree on feature ordering.
</details>

<details><summary>Solution</summary>

```python
from collections import Counter
from scipy.stats import spearmanr

# LEAR: count how many slots select each feature
selected = lear.selected_features()
feature_names = list(X.columns)
lear_freq = Counter()
for slot_feats in selected.values():
    lear_freq.update(slot_feats)

# GBT: feature importance
fi = gbt.feature_importance()

# Merge into a single DataFrame
comparison = fi.copy()
comparison["lear_freq"] = comparison["feature"].map(
    lambda f: lear_freq.get(f, 0)
)

print(comparison.to_string(index=False))

# Spearman rank correlation
rho, pval = spearmanr(comparison["importance"], comparison["lear_freq"])
print(f"\nSpearman rho: {rho:.3f}  (p={pval:.4f})")

if rho > 0.7:
    print("Strong agreement: both models find similar features important.")
elif rho > 0.3:
    print("Moderate agreement: some shared, some different priorities.")
else:
    print("Weak agreement: the models emphasise different features.")
    print("This is expected -- GBT exploits nonlinear interactions")
    print("that LASSO cannot capture with linear terms.")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(comparison["importance"], comparison["lear_freq"],
           s=80, color="steelblue")
for _, row in comparison.iterrows():
    ax.annotate(row["feature"], (row["importance"], row["lear_freq"]),
                fontsize=9, ha="left", va="bottom")
ax.set_xlabel("GBT importance (split gain)")
ax.set_ylabel("LEAR selection frequency (out of 48 slots)")
ax.set_title(f"Feature importance: GBT vs LEAR (Spearman rho={rho:.2f})")
fig.tight_layout()
plt.show()
```
</details>

In [ ]:
# Your analysis here

### Exercise 2: Linear baseline for the NN

How much does nonlinearity buy? Train a trivial linear network (single
linear layer, no hidden layers, no activation) with the same pinball loss.
Compare its CRPS and MAE to the full DayAheadQuantileNet.

<details><summary>Hint 1</summary>
Build a simple model: <code>nn.Linear(n_features, HORIZON * n_quantiles)</code>,
then reshape to <code>(batch, horizon, n_quantiles)</code> and sort along
the quantile dimension.
</details>

<details><summary>Hint 2</summary>
Train with the same optimizer, loss, and early stopping. Use fewer epochs
since the model is simpler and converges faster.
</details>

<details><summary>Hint 3</summary>
The gap between the linear net and the deep net tells you how much the
hidden layers contribute. If the gap is small, the problem might be
mostly linear (or the deep net is underfitting).
</details>

<details><summary>Solution</summary>

```python
import torch
import torch.nn as nn


class LinearQuantileNet(nn.Module):
    """Single linear layer -- no hidden units, no activation."""

    def __init__(self, n_features, horizon=48, n_quantiles=7):
        super().__init__()
        self.horizon = horizon
        self.n_quantiles = n_quantiles
        self.linear = nn.Linear(n_features, horizon * n_quantiles)

    def forward(self, x):
        raw = self.linear(x).view(-1, self.horizon, self.n_quantiles)
        return torch.sort(raw, dim=-1).values


torch.manual_seed(SEED)
linear_net = LinearQuantileNet(n_features, HORIZON, n_quantiles).to(device)
opt_lin = torch.optim.Adam(linear_net.parameters(), lr=1e-3)

n_params_lin = sum(p.numel() for p in linear_net.parameters())
print(f"Linear model parameters: {n_params_lin:,}")
print(f"Full NN parameters:      {n_params:,}")

# Re-create tensors for training
X_fit_t2 = torch.tensor(X_fit.numpy() if hasattr(X_fit, 'numpy') else X_train_daily[:n_fit],
                         dtype=torch.float32)
y_fit_t2 = torch.tensor(y_fit.numpy() if hasattr(y_fit, 'numpy') else y_train_daily[:n_fit],
                         dtype=torch.float32)
train_ds2 = TensorDataset(X_fit_t2, y_fit_t2)
train_loader2 = DataLoader(
    train_ds2, batch_size=64, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

best_lin_loss = float("inf")
best_lin_state = None
wait_lin = 0

for epoch in range(200):
    linear_net.train()
    for xb, yb in train_loader2:
        xb, yb = xb.to(device), yb.to(device)
        pred = linear_net(xb)
        loss = pinball_loss_fn(pred, yb.unsqueeze(-1), quantiles_t)
        opt_lin.zero_grad()
        loss.backward()
        opt_lin.step()

    linear_net.eval()
    with torch.no_grad():
        vp = linear_net(X_val_t)
        vl = pinball_loss_fn(vp, y_val_t.unsqueeze(-1), quantiles_t).item()

    if vl < best_lin_loss:
        best_lin_loss = vl
        best_lin_state = {
            k: v.cpu().clone() for k, v in linear_net.state_dict().items()
        }
        wait_lin = 0
    else:
        wait_lin += 1
        if wait_lin >= patience:
            print(f"Linear net early stopping at epoch {epoch + 1}")
            break

linear_net.load_state_dict(best_lin_state)
linear_net.to(device)

# Predict on test
linear_net.eval()
with torch.no_grad():
    lin_pred_asinh = linear_net(X_test_t).cpu().numpy()
lin_pred_flat = np.sinh(lin_pred_asinh).reshape(-1, n_quantiles)

# Actual targets (flattened)
y_test_flat = np.sinh(y_test_daily).flatten()

# Score
median_idx_q = QUANTILES.index(0.5)
mae_lin = mae(y_test_flat, lin_pred_flat[:, median_idx_q])
crps_lin = crps(y_test_flat, lin_pred_flat, np.array(QUANTILES))

print(f"\n{'Model':<20} {'MAE ($/MWh)':>12} {'CRPS ($/MWh)':>13}")
print("-" * 47)
print(f"{'Linear net':<20} ${mae_lin:>10.2f} ${crps_lin:>11.2f}")
print(f"{'Deep net (3-layer)':<20} ${mae_nn:>10.2f} ${crps_nn:>11.2f}")
print()
print(f"Nonlinearity buys:")
print(f"  {(1 - mae_nn / mae_lin) * 100:.1f}% MAE improvement")
print(f"  {(1 - crps_nn / crps_lin) * 100:.1f}% CRPS improvement")
```
</details>

In [ ]:
# Your analysis here

### Exercise 3: Capture ratio vs CRPS

Plot capture ratio against CRPS for all models that have both metrics.
Is the relationship monotonic -- does a lower CRPS always translate to
a higher capture ratio?

<details><summary>Hint 1</summary>
You need both CRPS and capture ratio for each model. LEAR and naive
lack CRPS, so this plot covers GBT and NN. For each, you have both
median and mean dispatch strategies (same CRPS, different CR), giving
four points.
</details>

<details><summary>Hint 2</summary>
Use a scatter plot with model names as annotations. If the relationship
is monotonic, points form a clear downward-sloping trend (lower CRPS =
higher CR).
</details>

<details><summary>Hint 3</summary>
Think about why they might NOT be monotonic: CRPS weights all quantiles
equally, but dispatch only uses the median (or mean). A model with
better tail quantiles (q0.05, q0.95) improves CRPS but may not change
the dispatch decision at all.
</details>

<details><summary>Solution</summary>

```python
# Collect (CRPS, capture ratio, label) for all model/strategy combos
scatter_data = [
    {"model": "GBT (median)", "CRPS": crps_gbt, "CR": cr["GBT_median"]},
    {"model": "GBT (mean)",   "CRPS": crps_gbt, "CR": cr["GBT_mean"]},
    {"model": "NN (median)",  "CRPS": crps_nn,  "CR": cr["NN_median"]},
    {"model": "NN (mean)",    "CRPS": crps_nn,  "CR": cr["NN_mean"]},
]
scat_df = pd.DataFrame(scatter_data)

fig, ax = plt.subplots(figsize=(8, 6))
colors_scat = ["steelblue", "royalblue", "darkorange", "goldenrod"]
for i, row in scat_df.iterrows():
    ax.scatter(row["CRPS"], row["CR"] * 100, s=120, color=colors_scat[i], zorder=5)
    ax.annotate(
        row["model"], (row["CRPS"], row["CR"] * 100),
        textcoords="offset points", xytext=(10, 5), fontsize=10,
    )

ax.set_xlabel("CRPS ($/MWh) -- lower is better")
ax.set_ylabel("Capture ratio (%) -- higher is better")
ax.set_title("Probabilistic accuracy vs dispatch value")
fig.tight_layout()
save_fig(fig, "crps_vs_capture_ratio")
plt.show()

print("Key insight: the relationship is NOT strictly monotonic.")
print("")
print("CRPS rewards calibrated quantiles across the full distribution,")
print("while capture ratio depends on the price signal (median or mean)")
print("that drives the dispatch LP.")
print("")
print("A model can improve CRPS through better tail quantiles without")
print("changing dispatch behaviour. Conversely, the mean-of-fan strategy")
print("can earn more than median dispatch by upweighting spike risk,")
print("even though the underlying CRPS is identical.")
```
</details>

In [ ]:
# Your analysis here

---
## Summary

- **LightGBM quantile regression** captures nonlinear feature interactions
  that the LEAR misses, typically improving both MAE and CRPS. Feature
  importance highlights which inputs drive the GBT's decisions.
- **DayAheadQuantileNet** learns the joint daily price profile, producing
  all 48 half-hour quantiles from a single feature vector. Training with
  pinball loss and early stopping prevents overfitting.
- **Quantile fan plots** reveal where models are confident vs uncertain.
  Volatile days produce wider fans; the actual price should fall within
  the outer quantiles most of the time.
- **Dispatch comparison** shows that forecast improvement translates to
  battery revenue, but the relationship between CRPS and capture ratio
  is not perfectly monotonic.
- **Median vs mean dispatch**: using the mean of the quantile fan can
  outperform the median when the distribution is skewed (positive skew
  in electricity prices means the mean is higher, encouraging more
  aggressive discharge during anticipated spikes).
- The GBT and NN both improve on the LEAR baseline, setting the stage
  for ensemble and conformal methods in notebooks 08--09.

---
## Report

In [ ]:
# Write report to outputs/reports/
report_dir = repo_root() / "outputs" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

report = f"""# Notebook 07: Machine Learning Models -- Report

Region: {REGION}
Train:  {train_start} to {train_end}
Test:   {test_start} to {test_end}

## Forecast accuracy

| Model | MAE ($/MWh) | rMAE | CRPS ($/MWh) |
|-------|-------------|------|--------------|
| Naive | {mae_naive:.2f} | 1.000 | -- |
| LEAR  | {mae_lear:.2f} | {rmae_lear:.3f} | -- |
| GBT   | {mae_gbt:.2f} | {rmae_gbt:.3f} | {crps_gbt:.2f} |
| NN    | {mae_nn:.2f} | {rmae_nn:.3f} | {crps_nn:.2f} |

## Battery dispatch

Battery: {cfg['battery']['power_mw']} MW / {cfg['battery']['duration_hours']} hr,
         {cfg['battery']['efficiency_roundtrip']:.0%} efficiency,
         {cfg['battery']['max_cycles_per_day']} max cycles/day

| Model         | Total revenue ($) | Capture ratio |
|---------------|-------------------|---------------|
| Naive         | {sum(dispatch['naive']['revenue']):,.0f} | {cr.get('naive', 0):.1%} |
| LEAR          | {sum(dispatch['LEAR']['revenue']):,.0f} | {cr.get('LEAR', 0):.1%} |
| GBT (median)  | {sum(dispatch['GBT_median']['revenue']):,.0f} | {cr.get('GBT_median', 0):.1%} |
| GBT (mean)    | {sum(dispatch['GBT_mean']['revenue']):,.0f} | {cr.get('GBT_mean', 0):.1%} |
| NN (median)   | {sum(dispatch['NN_median']['revenue']):,.0f} | {cr.get('NN_median', 0):.1%} |
| NN (mean)     | {sum(dispatch['NN_mean']['revenue']):,.0f} | {cr.get('NN_mean', 0):.1%} |
| Perfect       | {total_perfect:,.0f} | 100.0% |

## NN training

- Architecture: MLP (256, 128, 64) -> {HORIZON} x {n_quantiles} outputs
- Parameters: {n_params:,}
- Best epoch: {best_epoch + 1} (val pinball loss: {best_val_loss:.4f})
- Device: {{device}}

## Key findings

1. Both GBT and NN improve on LEAR for point and probabilistic accuracy.
2. Mean-of-fan dispatch can outperform median dispatch by upweighting spike risk.
3. A lower CRPS does not guarantee a higher capture ratio -- dispatch value
   depends on intra-day price ranking, not just calibrated quantile coverage.
"""

report_path = report_dir / "07_machine_learning.md"
report_path.write_text(report)
print(f"Report written to {report_path}")